# Import

In [ ]:
import os
import numpy as np
from beir import util, LoggingHandler
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.search.lexical.elastic_search import ElasticSearch
from beir.retrieval.models import SentenceBERT

In [ ]:
dataset = "scifact"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

In [ ]:
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# Dense Retrieval (Semantic)

In [ ]:
dense_model = DRES(SentenceBERT("sentence-transformers/all-MiniLM-L6-v2"), batch_size=64)
dense_retriever = EvaluateRetrieval(dense_model, score_function="cos_sim")
dense_results = dense_retriever.retrieve(corpus, queries)

In [ ]:
dense_results

# Sparse Retrieval (BM25)

In [ ]:
# 1. Обязательный импорт нового класса для BEIR 2.0+
from beir.retrieval.search.lexical.elastic_search import ElasticSearch
from beir.retrieval.evaluation import EvaluateRetrieval

# 2. Настройки подключения к твоему поднятому Docker-контейнеру
hostname = "http://localhost:9200" # В новых версиях BEIR лучше указывать полный URL
index_name = dataset # scifact

# 3. Инициализация и поиск
# initialize=True заставит BEIR создать индекс в Elastic и залить туда corpus
sparse_model = ElasticSearch(index_name=index_name, hostname=hostname, initialize=True)
sparse_retriever = EvaluateRetrieval(sparse_model)

print("Starting BM25 retrieval via Elasticsearch...")
sparse_results = sparse_retriever.retrieve(corpus, queries)
print("Sparse retrieval completed!")